# CUDA FFT Convolution Benchmark

**重要**: 运行前请确保:
1. Runtime → Change runtime type → GPU
2. 上传你的 needle 代码到 Colab

## 1. 检查 GPU 环境

In [ ]:
# 检查 GPU
!nvidia-smi

# 检查 CUDA
!nvcc --version

## 2. 上传代码

选择以下方法之一:

### 方法 A: 从 GitHub 克隆

In [ ]:
# 如果你的代码在 GitHub 上
!git clone https://github.com/YOUR_USERNAME/needle.git
%cd needle

### 方法 B: 从本地上传

1. 点击左侧文件图标
2. 上传整个 needle 文件夹（包含 src/, python/, CMakeLists.txt 等）
3. 然后运行下面的单元格

In [ ]:
# 切换到 needle 目录（调整路径）
%cd /content/needle

# 验证文件存在
!ls -la

## 3. 安装依赖并编译

In [ ]:
# 安装依赖
!pip install pybind11 numpy -q

In [ ]:
# 清理并编译
!make clean
!make

In [ ]:
# 验证编译成功
!ls python/needle/backend_ndarray/*.so

## 4. 运行 CUDA FFT Benchmark

In [ ]:
# 设置环境变量
import os
os.environ['NEEDLE_FFT_IMPL'] = 'cuda'

# 运行 benchmark
!python colab_cuda_fft_benchmark.py

## 5. (可选) 手动测试 CUDA FFT

In [ ]:
import sys
sys.path.insert(0, './python')

import os
os.environ['NEEDLE_FFT_IMPL'] = 'cuda'

import numpy as np
import needle
from needle import Tensor
import needle.ops as ops

# 创建 CUDA device
device = needle.cuda()
print(f"✓ CUDA device: {device}")

# 测试 FFT
x = Tensor(np.random.randn(2, 3, 64, 64).astype(np.float32), device=device)
print(f"✓ Input shape: {x.shape}")

# FFT
fft_result = ops.fft(x, dim=-1, norm="backward")
real_part = ops.tuple_get_item(fft_result, 0)
imag_part = ops.tuple_get_item(fft_result, 1)

print(f"✓ FFT real shape: {real_part.shape}")
print(f"✓ FFT imag shape: {imag_part.shape}")

# IFFT
x_reconstructed = ops.ifft(real_part, imag_part, dim=-1, norm="backward")
print(f"✓ IFFT shape: {x_reconstructed.shape}")

# 验证精度
error = np.max(np.abs(x.numpy() - x_reconstructed.numpy()))
print(f"\nReconstruction error: {error:.2e}")

if error < 1e-4:
    print("✓ CUDA FFT/IFFT working correctly!")
else:
    print(f"⚠ Large error detected: {error}")

## 6. (可选) 自定义性能测试

In [ ]:
import time
import sys
sys.path.insert(0, './python')

import numpy as np
import needle
from needle import Tensor
import needle.nn as nn

device = needle.cuda()

# 测试配置 - 可以修改这些参数
batch_size = 1
in_channels = 3
out_channels = 16
H, W = 256, 256
kernel_size = 21

print(f"Testing: {H}×{W}, K={kernel_size}, Batch={batch_size}")

# 创建输入
x = Tensor(np.random.randn(batch_size, in_channels, H, W).astype(np.float32), device=device)

# 空间卷积
conv_spatial = nn.Conv(in_channels, out_channels, kernel_size, device=device)

# Warmup
for _ in range(3):
    _ = conv_spatial(x)

# Benchmark
start = time.time()
for _ in range(10):
    y = conv_spatial(x)
spatial_time = (time.time() - start) / 10 * 1000

print(f"\nSpatial convolution: {spatial_time:.2f} ms/iter")
print(f"Output shape: {y.shape}")

# 注意: FFT 卷积需要使用 FrequencyConv2D 类（在 colab_cuda_fft_benchmark.py 中）

## 结果分析

预期结果：
- **小图像/小卷积核**: 空间卷积更快
- **大图像/大卷积核**: FFT 卷积应该更快（特别是 K≥15 时）

如果看到 FFT 在大卷积核上有加速，说明 CUDA FFT 实现成功！ 🎉